# 03 — Pollinator Occurrences (Full Scale)

Builds the pollinator existence matrix P and activity curves a_curves
from the full corrected pollinator dataset.

**Data source:** `pollinator_observations_v2.csv` (25,466 species, CONUS, 2013–2026)

**Outputs:**
- `stage4_P_existence_corrected.csv` — binary pollinator existence matrix P (24,939 species × 3,162 bins)
- `stage4_Vp_corrected.csv` — PCA 15D pollinator embedding Vp (46.2% variance explained)
- `a_curves_corrected.csv` — normalized 52-week activity curves (25,466 species)

**Critical note on data quality:** GBIF pollinator observations are opportunistic
citizen science records, primarily sourced from iNaturalist. Observation density
in any spatial bin or week reflects recorder activity, not genuine pollinator
presence or phenological timing. This bias has two consequences:

1. **Spatial bias (P matrix):** Bins near human population centers are
   overrepresented. Vp's 46.2% variance explained in 15 components partly
   reflects human geography rather than ecological distribution structure.

2. **Temporal bias (a_curves):** Weekly observation histograms reflect when
   observers were active outdoors, not when pollinators were active. This
   is the core asymmetry finding: enriching the model with GBIF-derived
   temporal signal on the pollinator side consistently degrades performance
   (see experiments). SDM-derived activity curves (Dan Cher) are the
   intended replacement — model-predicted curves independent of observation effort.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from pathlib import Path
import gc

# ── Paths ──────────────────────────────────────────────────────────────────
BASE       = Path("/scratch/ariana.l")
POLL_OBS   = BASE / "Plant Pollinator Initial Analysis" / "pollinator_observations_v2.csv"
F_COMMON   = BASE / "Stage 4 Link Prediction Model" / "stage4_F_existence_phenofield.csv"
OUT_DIR    = BASE / "New Stage 4 Link Prediction Model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

P_OUT      = OUT_DIR / "stage4_P_existence_corrected.csv"
VP_OUT     = OUT_DIR / "stage4_Vp_corrected.csv"
A_CURVES_OUT = OUT_DIR / "a_curves_corrected.csv"

# ── Constants ───────────────────────────────────────────────────────────────
BIN_SIZE   = 0.5
CHUNK_SIZE = 500_000

print("Paths OK")

In [ ]:
# Load F matrix to extract common CONUS bins
print("Loading F existence matrix to extract common bins...")
F_df = pd.read_csv(F_COMMON, index_col=0)
common_bins = list(F_df.columns)
common_bins_set = set(common_bins)
print(f"  F matrix shape : {F_df.shape}")
print(f"  Common bins    : {len(common_bins):,}")
print(f"  Sample bins    : {common_bins[:3]}")

In [ ]:
# Build P existence matrix by streaming pollinator_observations_v2.csv
# Bin format: "lat_lon" e.g. "34.5_-120.0"
print("Streaming pollinator observations → P existence matrix...")

def snap_bin(lat, lon, size=0.5):
    lat_bin = round(np.floor(lat / size) * size, 1)
    lon_bin = round(np.floor(lon / size) * size, 1)
    return f"{lat_bin}_{lon_bin}"

presence = {}
reader = pd.read_csv(
    POLL_OBS,
    usecols=["pollinator_species", "lat", "lon"],
    chunksize=CHUNK_SIZE,
    low_memory=False,
)

chunk_count = 0
for chunk in reader:
    chunk = chunk.dropna(subset=["pollinator_species", "lat", "lon"])
    chunk["bin"] = [
        snap_bin(lat, lon)
        for lat, lon in zip(chunk["lat"], chunk["lon"])
    ]
    chunk = chunk[chunk["bin"].isin(common_bins_set)]
    for row in chunk[["pollinator_species", "bin"]].itertuples(index=False):
        if row.pollinator_species not in presence:
            presence[row.pollinator_species] = set()
        presence[row.pollinator_species].add(row.bin)
    chunk_count += 1
    if chunk_count % 10 == 0:
        print(f"  ...{chunk_count} chunks, {len(presence):,} species so far")
    del chunk
    gc.collect()

print(f"\nDone. {len(presence):,} pollinator species with ≥1 CONUS bin.")

In [ ]:
# Convert presence dict → binary DataFrame → save P matrix
print("Building P existence matrix...")

all_species = sorted(presence.keys())
rows = []
for sp in all_species:
    row = dict.fromkeys(common_bins, 0)
    for b in presence[sp]:
        row[b] = 1
    rows.append(row)

P_df = pd.DataFrame(rows, index=all_species, columns=common_bins)
print(f"  P matrix shape : {P_df.shape}")
print(f"  Sparsity       : {1 - P_df.values.mean():.4f}")

P_df.to_csv(P_OUT)
print(f"  Saved → {P_OUT}")

del rows
gc.collect()

In [ ]:
# PCA 15D on P matrix → Vp
print("Fitting PCA on P matrix...")

pca_p = PCA(n_components=15, svd_solver='randomized', random_state=42)
Vp_arr = pca_p.fit_transform(P_df.values)

explained = pca_p.explained_variance_ratio_.sum()
print(f"  Variance explained (15 components) : {explained:.4f}")
# Expected: ~0.4624
# Lower than plant-side Vf (0.399) reflects observation-density bias
# compressing geographic distribution structure

Vp_df = pd.DataFrame(Vp_arr, index=P_df.index,
                      columns=[f"PC{i+1}" for i in range(15)])
Vp_df.to_csv(VP_OUT)
print(f"  Saved → {VP_OUT}")
print(f"  Vp shape : {Vp_df.shape}")

In [ ]:
# Build a_curves: normalized 52-week activity histograms
# Week formula: (doy - 1) // 7, clipped to [0, 51]
#
# IMPORTANT: these curves reflect GBIF observation timing, not true
# pollinator activity. Peaks in a_curves correspond to weeks when
# iNaturalist users were most active outdoors, not biological peaks.
# Use SDM-derived curves for bias-corrected temporal signal.

print("Building a_curves from pollinator observations...")

activity = {}
reader = pd.read_csv(
    POLL_OBS,
    usecols=["pollinator_species", "doy"],
    chunksize=CHUNK_SIZE,
    low_memory=False,
)

chunk_count = 0
for chunk in reader:
    chunk = chunk.dropna(subset=["pollinator_species", "doy"])
    chunk["week"] = ((chunk["doy"].astype(int) - 1) // 7).clip(0, 51)
    for sp, grp in chunk.groupby("pollinator_species"):
        counts = grp["week"].value_counts()
        if sp not in activity:
            activity[sp] = np.zeros(52)
        for week, cnt in counts.items():
            activity[sp][int(week)] += cnt
    chunk_count += 1
    if chunk_count % 10 == 0:
        print(f"  ...{chunk_count} chunks, {len(activity):,} species so far")
    del chunk
    gc.collect()

print("Normalizing...")
records = []
for sp, arr in activity.items():
    total = arr.sum()
    if total > 0:
        arr = arr / total
    records.append([sp] + arr.tolist())

a_curves_df = pd.DataFrame(
    records,
    columns=["species"] + [f"w{i}" for i in range(52)]
).set_index("species")

a_curves_df.to_csv(A_CURVES_OUT)
print(f"  Saved → {A_CURVES_OUT}")
print(f"  a_curves shape : {a_curves_df.shape}")
# Expected: (25466, 52)